# K-Nearest Neighbors (K-NN) em datasets com class imbalance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from math import sqrt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)

## Implementação KNN from scratch

In [ ]:
class KNN:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X_train, y_train):
        self.x_train = X_train
        self.y_train = y_train

    def calculate_euclidean(self, sample1, sample2):
        distance = 0.0
        for i in range(len(sample1)):
            distance += (sample1[i] - sample2[i]) ** 2
        return sqrt(distance)

    def nearest_neighbors(self, test_sample):
        distances = []
        for i in range(len(self.x_train)):
            distances.append((self.y_train[i], self.calculate_euclidean(self.x_train[i], test_sample)))
        distances.sort(key=lambda x: x[1])
        neighbors = []
        k_eff = min(self.k, len(distances))
        for i in range(k_eff):
            neighbors.append(distances[i][0])
        return neighbors

    def predict(self, test_set):
        predictions = []
        for test_sample in test_set:
            neighbors = self.nearest_neighbors(test_sample)
            labels = [sample for sample in neighbors]
            prediction = max(labels, key=labels.count)
            predictions.append(prediction)
        return np.array(predictions)

## Datasets disponíveis e seleção para teste empírico

In [ ]:
data_dir = Path("class_imbalance/class_imbalance")
all_csv = sorted(data_dir.glob("*.csv"))

print(f"Total de datasets encontrados: {len(all_csv)}")
for p in all_csv[:10]:
    print("-", p.name)

MAX_DATASETS = 12
selected_paths = all_csv[:MAX_DATASETS]

print("\nDatasets selecionados:")
for p in selected_paths:
    print("-", p.name, "| existe:", p.exists())

## Funções auxiliares para análise e avaliação

In [ ]:
def infer_target_column(df):
    return df.columns[-1]

def class_distribution_report(y, dataset_name, plot=False):
    s = pd.Series(y)
    counts = s.value_counts(dropna=False)
    ratios = (counts / len(s)).round(4)

    print(f"\n=== Distribuição de classes: {dataset_name} ===")
    dist_df = pd.DataFrame({"count": counts, "ratio": ratios})
    print(dist_df)

    if len(counts) > 1:
        imbalance_ratio = counts.max() / counts.min()
        print(f"Imbalance ratio (majoritária/minoritária): {imbalance_ratio:.2f}")
    else:
        print("Apenas uma classe presente.")

    if plot:
        ax = counts.sort_index().plot(kind="bar", figsize=(6, 3), title=f"Distribuição - {dataset_name}")
        ax.set_xlabel("Classe")
        ax.set_ylabel("Contagem")
        plt.tight_layout()
        plt.show()

def evaluate_predictions(y_true, y_pred, average_labels=("macro", "weighted")):
    out = {}
    out["accuracy"] = accuracy_score(y_true, y_pred)
    out["balanced_accuracy"] = balanced_accuracy_score(y_true, y_pred)

    for avg in average_labels:
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=avg, zero_division=0
        )
        out[f"precision_{avg}"] = p
        out[f"recall_{avg}"] = r
        out[f"f1_{avg}"] = f1

    return out

def is_dataset_valid_for_stratified_split(y, test_size=0.25):
    s = pd.Series(y)
    counts = s.value_counts(dropna=False)
    if len(counts) < 2:
        return False, "Apenas uma classe."
    if counts.min() < 2:
        return False, "Classe com menos de 2 amostras (split estratificado inviável)."
    n_test = int(np.ceil(len(s) * test_size))
    if n_test < len(counts):
        return False, "Conjunto de teste pequeno demais para conter todas as classes."
    return True, 

## Avaliação empírica em múltiplos datasets desbalanceados

In [ ]:
results = []
skipped = []

for ds_path in selected_paths:
    if not ds_path.exists():
        print(f"Dataset não encontrado: {ds_path}")
        skipped.append({"dataset": ds_path.name, "reason": "Ficheiro não encontrado"})
        continue

    print("\n" + "=" * 80)
    print(f"Dataset: {ds_path.name}")

    try:
        df = pd.read_csv(ds_path)
        if df.shape[1] < 2:
            skipped.append({"dataset": ds_path.name, "reason": "Dataset sem colunas suficientes"})
            print("Ignorado: Dataset sem colunas suficientes")
            continue

        target_col = infer_target_column(df)
        X = df.drop(columns=[target_col]).copy()
        y = df[target_col].copy()

        non_null_mask = ~pd.isna(y)
        X = X.loc[non_null_mask]
        y = y.loc[non_null_mask]

        X = pd.get_dummies(X, drop_first=True)
        if X.isnull().any().any():
            X = X.fillna(X.median(numeric_only=True))
        X = X.apply(pd.to_numeric, errors="coerce").fillna(0.0)

        if y.dtype == "O" or str(y.dtype).startswith("category"):
            y = y.astype("category").cat.codes

        valid_split, reason = is_dataset_valid_for_stratified_split(y, test_size=0.25)
        if not valid_split:
            skipped.append({"dataset": ds_path.name, "reason": reason})
            print(f"Ignorado: {reason}")
            continue

        class_distribution_report(y, ds_path.name, plot=False)

        X_train, X_test, y_train, y_test = train_test_split(
            X.values, np.array(y), test_size=0.25, random_state=42, stratify=y
        )

        k_eff = min(5, len(X_train))
        if k_eff < 1:
            skipped.append({"dataset": ds_path.name, "reason": "Treino vazio"})
            print("Ignorado: treino vazio")
            continue

        sc = StandardScaler()
        X_train_sc = sc.fit_transform(X_train)
        X_test_sc = sc.transform(X_test)

        knn_scratch = KNN(k=k_eff)
        knn_scratch.fit(X_train_sc, y_train)
        pred_scratch = knn_scratch.predict(X_test_sc)
        met_scratch = evaluate_predictions(y_test, pred_scratch)
        met_scratch.update({"dataset": ds_path.name, "model": "knn_from_scratch"})
        results.append(met_scratch)

        knn_uniform = KNeighborsClassifier(
            n_neighbors=k_eff, metric="minkowski", p=2, weights="uniform"
        )
        knn_uniform.fit(X_train_sc, y_train)
        pred_uniform = knn_uniform.predict(X_test_sc)
        met_uniform = evaluate_predictions(y_test, pred_uniform)
        met_uniform.update({"dataset": ds_path.name, "model": "sklearn_knn_uniform"})
        results.append(met_uniform)

        knn_distance = KNeighborsClassifier(
            n_neighbors=k_eff, metric="minkowski", p=2, weights="distance"
        )
        knn_distance.fit(X_train_sc, y_train)
        pred_distance = knn_distance.predict(X_test_sc)
        met_distance = evaluate_predictions(y_test, pred_distance)
        met_distance.update({"dataset": ds_path.name, "model": "sklearn_knn_distance"})
        results.append(met_distance)

        print("\nConfusion Matrix (distance-weighted):")
        print(confusion_matrix(y_test, pred_distance))
        print("\nClassification Report (distance-weighted):")
        print(classification_report(y_test, pred_distance, zero_division=0))

    except Exception as e:
        skipped.append({"dataset": ds_path.name, "reason": f"Erro: {str(e)}"})
        print(f"Ignorado por erro: {e}")

## Tabela final de resultados e consolidação

In [ ]:
results_df = pd.DataFrame(results)

if not results_df.empty:
    metric_cols = [
        "accuracy", "balanced_accuracy",
        "precision_macro", "recall_macro", "f1_macro",
        "precision_weighted", "recall_weighted", "f1_weighted",
    ]

    display_df = results_df[["dataset", "model"] + metric_cols].sort_values(
        by=["dataset", "balanced_accuracy"], ascending=[True, False]
    )

    print("\n=== Resultados por dataset/modelo ===")
    print(display_df.to_string(index=False))

    ranking = (
        results_df.groupby("model")[["balanced_accuracy", "f1_macro", "f1_weighted"]]
        .mean()
        .sort_values(by=["balanced_accuracy", "f1_macro"], ascending=False)
        .reset_index()
    )

    print("\n=== Ranking médio global por modelo ===")
    print(ranking.to_string(index=False))
else:
    print("Sem resultados. Verifica os datasets selecionados.")

if skipped:
    skipped_df = pd.DataFrame(skipped)
    print("\n=== Datasets ignorados e motivo ===")
    print(skipped_df.to_string(index=False))